## ETAPA 1. Análisis exploratorio y consultas.

In [1]:
"""
Etapa 1: Análisis exploratorio y consultas sobre el conjunto de datos
de multas de circulación del Ayuntamiento de Madrid (Diciembre 2024).
"""

import io

import requests
import pandas as pd

### La descarga de datos

En este notebook vamos a estudiar únicamente los datos de diciembre  de 2024.
La URL que permite descargar dichos datos es la siguiente:

In [2]:
raiz = "https://datos.madrid.es/"
url_diciembre_24 = "dataset/210104-0-multas-circulacion-detalle/resource/210104-15-multas-circulacion-detalle-csv/download/210104-15-multas-circulacion-detalle-csv.csv"
url = raiz + url_diciembre_24.lstrip("/")

Podemos hacer la petición con request.get():

In [3]:
response = requests.get(url)
if response.status_code == 200:
    print(response.url)

https://datos.madrid.es/dataset/210104-0-multas-circulacion-detalle/resource/210104-15-multas-circulacion-detalle-csv/download/210104-15-multas-circulacion-detalle-csv.csv


In [4]:
# Accedemos al texto
content = io.BytesIO(response.content)
multas = pd.read_csv(content, sep =';', encoding = 'latin1')
multas.head()

,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y             
0,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA. ...,,,,            
1,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",,,,            
2,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACIÓN DE UN...,,,,            
3,LEVE,CL BRAVO MURILLO 24,12,2024,16.30,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA. ...,,,,            
4,LEVE,CL BRAVO MURILLO 16,12,2024,16.50,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",,,,            


### Explorar la estructura de los datos en el dataframe

El método `info` devuelve información acerca de la estructura de un dataframe. Ejecuta el método `info()` sobre el dataframe `multas` y comenta los resultados.
Fíjate en las columnas que representan la hora, el día y el año y observa su tipo de datos. ¿Cuántas filas se han cargado?. Fíjate en los valores distintos de `NaN` en cada una de las columnas.

In [5]:
multas.info()

<class 'pandas.DataFrame'>
RangeIndex: 249801 entries, 0 to 249800
Data columns (total 14 columns):
 #   Column                                                                                                                                                              Non-Null Count   Dtype  
---  ------                                                                                                                                                              --------------   -----  
 0   CALIFICACION                                                                                                                                                        249801 non-null  str    
 1   LUGAR                                                                                                                                                               249801 non-null  str    
 2   MES                                                                                                                    

### Resultados de la exploración del dataframe `multas` con `info()`

- **Dimensiones:** 249.801 filas y 14 columnas.
- **Valores nulos:** no hay valores nulos reconocidos (`NaN`, `NaT` o `<NA>`).
  - Posible incidencia: podrían estar codificados como cadenas vacías (`""`), espacios (`" "`) o etiquetas manuales (`"SIN DATO"`, `"N/A"`, `"-999"`).
- **Columnas temporales:**
  - `ANIO` y `MES` son `int64`, correcto para sus valores.
  - `HORA` es `float64` al interpretar el formato `HH.MM` como decimal.
  - No existe columna `DIA`.
- **Columnas de velocidad y geoespaciales:**
  - `VEL_LIMITE`, `VEL_CIRCULA`, `COORDENADA-X` y `COORDENADA-Y` se han leído como texto, probablemente por celdas vacías o caracteres no numéricos.

### EXPLORACIÓN DE LAS COLUMNAS
__a)__ ¿Cuántos valores distintos hay en la columna `CALIFICACION` del dataframe `multas`? Modifica los valores de dicha columna para que no contengan espacios en blanco. Puedes usar el método  `str.strip()` de la clase Series.

In [6]:
# Valores distintos columna CALIFICACION
multas['CALIFICACION'].unique()

<StringArray>
['LEVE      ', 'GRAVE     ', 'MUY GRAVE ']
Length: 3, dtype: str

In [7]:
# Eliminar espacios en blanco al inicio y al final de cada valor
multas['CALIFICACION'] = multas['CALIFICACION'].str.strip()

# Revisar tras aplicar el metodo str.strip()
multas['CALIFICACION'].unique()

<StringArray>
['LEVE', 'GRAVE', 'MUY GRAVE']
Length: 3, dtype: str

__b)__ ¿Cuántos valores distintos hay en las columnas `DESCUENTO`, `HECHO-BOL` y `DENUNCIANTE` del dataframe `multas`? Modifica los valores de dichas columnas para que no contengan espacios en blanco.

In [8]:
# Nº de valores distintos columnas DESCUENTO, HECHO-BOL Y DENUNCIANTE (contando NaN si existiera)
multas[['DESCUENTO', 'HECHO-BOL', 'DENUNCIANTE']].nunique(dropna=False)

DESCUENTO        1
HECHO-BOL      345
DENUNCIANTE      6
dtype: int64

In [9]:
# Valores distintos de cada columna
print(multas['DESCUENTO'].unique())
print(multas['HECHO-BOL'].unique()[:10].tolist()) # 10 primeros a modo de ejemplo. Total:345
print(multas['DENUNCIANTE'].unique().tolist())

<StringArray>
['SI']
Length: 1, dtype: str
['ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.                                                                                       ', 'ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓN, EN LUGAR HABILITADO PARA EL ESTACIONAMIENTO CON LIMITACIÓN HORARIA.         ', 'ESTACIONAR OBSTACULIZANDO LA UTILIZACIÓN DE UN VADO SEÑALIZADO CORRECTAMENTE.                                                ', 'ESTACIONAR SIN TÍTULO HABILITANTE EN RESERVA DE CARGA Y DESCARGA DE MERCANCIAS                                               ', 'ESTACIONAR SOBRE O JUNTO A ELEMENTOS CANALIZADORES DEL TRÁFICO OBSTACULIZANDO GRAVEMENTELA CIRCULACIÓN.                      ', 'ESTACIONAR EN LUGAR PROHIBIDO.                                                                                               ', 'ESTACIONAR CON AUTORIZACIÓN EN LUGAR HABILITADO PARA EL ESTACIONAMIENTO CON LIMITACIÓNHORARIA,REBASANDO EL TIEMPO MÁXIMO     ', 'ESTACIONAR CON UN TITULO QUE NO HABILITA PARA LA RES

In [10]:
# Limpiar espacios en blanco en las tres columnas
cols = ['DESCUENTO', 'HECHO-BOL', 'DENUNCIANTE']
multas[cols] = multas[cols].apply(lambda x: x.str.strip())

In [11]:
# Revisar valores distintos de cada columna tras aplicar metodo str.strip()
print(multas['DESCUENTO'].unique())
print(multas['HECHO-BOL'].unique()[:10].tolist()) # 10 primeros a modo de ejemplo. Total:345
print(multas['DENUNCIANTE'].unique().tolist())

<StringArray>
['SI']
Length: 1, dtype: str
['ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.', 'ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓN, EN LUGAR HABILITADO PARA EL ESTACIONAMIENTO CON LIMITACIÓN HORARIA.', 'ESTACIONAR OBSTACULIZANDO LA UTILIZACIÓN DE UN VADO SEÑALIZADO CORRECTAMENTE.', 'ESTACIONAR SIN TÍTULO HABILITANTE EN RESERVA DE CARGA Y DESCARGA DE MERCANCIAS', 'ESTACIONAR SOBRE O JUNTO A ELEMENTOS CANALIZADORES DEL TRÁFICO OBSTACULIZANDO GRAVEMENTELA CIRCULACIÓN.', 'ESTACIONAR EN LUGAR PROHIBIDO.', 'ESTACIONAR CON AUTORIZACIÓN EN LUGAR HABILITADO PARA EL ESTACIONAMIENTO CON LIMITACIÓNHORARIA,REBASANDO EL TIEMPO MÁXIMO', 'ESTACIONAR CON UN TITULO QUE NO HABILITA PARA LA RESERVA DE CARGA Y DESCARGA DE MERCANCÍA DONDE HA ESTACIONADO.', 'ESTACIONAR SUPERANDO ELTIEMPO PERMITIDO EN ZONA DE CARGAY DESCARGA DURANTE LAS HORAS FIJADAS PARA SU USO.', 'ESTACIONAR EN LUGAR PROHIBIDO. VEHÍCULO DE CATEGORÍA AMBIENTAL A EN ZONA SER.']
['SER', 'AGENTES DE MOVILIDAD', 'POLICIA MUNICIPAL', 'MEDIOS 

__c)__ Muestra el listado de columnas del dataframe.

In [12]:
multas.columns.tolist()

['CALIFICACION',
 'LUGAR',
 'MES',
 'ANIO',
 'HORA',
 'IMP_BOL',
 'DESCUENTO',
 ' PUNTOS',
 'DENUNCIANTE',
 'HECHO-BOL',
 'VEL_LIMITE',
 'VEL_CIRCULA ',
 'COORDENADA-X',
 'COORDENADA-Y                                                                                                                                          \x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00']

In [13]:
# Columnas con caracteres nulos o espacios sobrantes en el nombre
columnas_afectadas = [c for c in multas.columns if c != c.replace('\x00', '').strip()]

pd.DataFrame({
    'nombre_normalizado': [c.replace('\x00', '').strip() for c in columnas_afectadas],
    'longitud_original': [len(c) for c in columnas_afectadas],
    'espacios_sobrantes': [len(c.replace('\x00', '')) - len(c.replace('\x00', '').strip()) for c in columnas_afectadas],
    'caracteres_nulos': [c.count('\x00') for c in columnas_afectadas],
})

,nombre_normalizado,longitud_original,espacios_sobrantes,caracteres_nulos
0,PUNTOS,7,1,0
1,VEL_CIRCULA,12,1,0
2,COORDENADA-Y,162,138,12


¿Cuál es el nombre de la columna que representa la velocidad de circulación registrada? Fíjate en el espacio en blanco al final.

>`'VEL_CIRCULA '` con 1 espacio al final.

¿Cuál es el nombre de la columna que representa la coordenada Y de la ubicación geográfica de donde se produce la infracción? Fíjate en la cantidad de espacios en blanco al final.

> `'COORDENADA-Y'` con 138 espacios al final y 12 valores nulos (\x00)

Muestra la serie de datos correspondiente a la coordenada Y.

In [14]:
# squeeze() convierte el DataFrame de una columna en una Serie. Permite usar el filtro like para seleccionar el complejo nombre actual de la columna
print(type(multas.filter(like='COORDENADA-Y').squeeze()))
multas.filter(like='COORDENADA-Y').squeeze()

<class 'pandas.Series'>


0                                
1                                
2                                
3                                
4                                
                   ...           
249796                           
249797                           
249798                           
249799                           
249800                           
Name: COORDENADA-Y                                                                                                                                                      , Length: 249801, dtype: str

Usa el método  `rename` de los dataframes para normalizar los nombres de las columnas del dataframe multas, de forma que los nuevos nombres no tengan espacios en blanco al final. Consulta la ayuda del método `rename`; ésta te propone el uso de métodos de la clase `str` a cada elemento del índice de las columnas (o de las filas). En nuestro caso, para eliminar espacios en blanco usaremos el método `str.strip()` sobre el índice de las columnas.

In [15]:
# Eliminar los caracteres nulos (\x00) y luego los espacios en blanco
multas = multas.rename(columns=lambda x: x.replace('\x00', '').strip())

In [16]:
multas.columns.tolist()

['CALIFICACION',
 'LUGAR',
 'MES',
 'ANIO',
 'HORA',
 'IMP_BOL',
 'DESCUENTO',
 'PUNTOS',
 'DENUNCIANTE',
 'HECHO-BOL',
 'VEL_LIMITE',
 'VEL_CIRCULA',
 'COORDENADA-X',
 'COORDENADA-Y']

__d)__ ¿Observa las columnas relativas a la velocidad. ¿De qué tipo son?.
¿Cuántos valores distintos hay en las columnas `VEL_LIMITE`, `VEL_CIRCULA`. Por último, estudia la función `pd.to_numeric` de pandas y úsala para transformar el tipo de las columnas  anteriores a tipo numérico.

In [17]:
# Tipo (dtypes) - Filas (size) -  Valores (count) - Valores distintos (nunique)
multas[['VEL_LIMITE', 'VEL_CIRCULA']].agg(['dtypes','size','count','nunique']).T

,dtypes,size,count,nunique
VEL_LIMITE,str,249801,249801,7
VEL_CIRCULA,str,249801,249801,101


#### Conversión a tipo numérico con `pd.to_numeric()`

`pd.to_numeric()` convierte los valores de una columna de texto a `int64` o `float64`.

- **Limpieza previa:** las columnas contienen caracteres nulos (`\x00`) y espacios sobrantes que impiden la conversión. Se eliminan con `str.replace('\x00', '')` y `str.strip()` antes de convertir; si no, `to_numeric` no reconocería ni siquiera los valores válidos.
- **`errors='coerce'`:** sin este argumento, un valor no convertible (por ejemplo, `"SIN DATO"` o una celda vacía) lanza un error y detiene la ejecución. Con él, esos valores pasan a `NaN`. Como `NaN` no es compatible con enteros, la columna se convierte a `float64` (`70` pasa a `70.0`).
- **`astype('Int64')`:** al ser las velocidades números enteros, se fuerza el tipo entero nullable de pandas, que admite `<NA>` y mantiene los valores como enteros.

In [18]:
# Limpiar \x00 y espacios, convertir a numérico y forzar a entero nullable (Int64)
cols = ['VEL_LIMITE', 'VEL_CIRCULA']

multas[cols] = multas[cols].apply(
    lambda s: pd.to_numeric(
        s.str.replace('\x00', '', regex=False).str.strip(),
        errors='coerce'
    )
).astype('Int64')

In [19]:
# Resumen tablas velocidad tras la conversión (nunique ignora los nulos reconocidos NaN)
multas[['VEL_LIMITE', 'VEL_CIRCULA']].agg(['dtypes', 'size','count','nunique']).T

,dtypes,size,count,nunique
VEL_LIMITE,Int64,249801,22182,6
VEL_CIRCULA,Int64,249801,22182,100


In [20]:
# Mostrar las primeras filas con datos de velocidad
multas[['VEL_LIMITE', 'VEL_CIRCULA']].dropna().head()

,VEL_LIMITE,VEL_CIRCULA
36955,50,74
36956,50,79
37114,60,100
37115,50,73
37116,60,103


__e)__ ¿Observa las columnas relativas a la geolocalización. ¿De qué tipo son?. Usa la función `pd.to_numeric` de pandas para transformar el tipo de las columnas  `COORDENADA-X`, `COORDENADA-Y` a tipo numérico.

In [21]:
# Tipo (dtypes) - Filas (size) -  Valores (count) - Valores distintos (nunique)
multas[['COORDENADA-X', 'COORDENADA-Y']].agg(['dtypes','size','count','nunique']).T

,dtypes,size,count,nunique
COORDENADA-X,str,249801,249801,2455
COORDENADA-Y,str,249801,249801,2500


In [22]:
# Limpiar caracteres nulos (\x00), espacios y convertir las coordenadas a tipo numérico (float).
cols = ['COORDENADA-X', 'COORDENADA-Y']

multas[cols] = multas[cols].apply(
    lambda s: pd.to_numeric(
        s.str.replace('\x00', '', regex=False).str.strip(),
        errors='coerce'
    )
)

In [23]:
# Resumen de las tablas de geolocalización tras la conversion
multas[['COORDENADA-X', 'COORDENADA-Y']].agg(['dtypes','size','count','nunique']).T

,dtypes,size,count,nunique
COORDENADA-X,float64,249801,123882,2454
COORDENADA-Y,float64,249801,123882,2499


In [24]:
# Mostrar las primeras filas que contienen coordenadas
multas[['COORDENADA-X', 'COORDENADA-Y']].dropna().head()

,COORDENADA-X,COORDENADA-Y
1012,4407.67,44748.29
1013,4407.67,44748.29
1014,4407.67,44748.29
1015,4407.67,44748.29
1016,4397.66,44751.88


El dataframe obtenido después de aplicar todas las transformaciones anteriores

In [25]:
multas.info()

<class 'pandas.DataFrame'>
RangeIndex: 249801 entries, 0 to 249800
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   CALIFICACION  249801 non-null  str    
 1   LUGAR         249801 non-null  str    
 2   MES           249801 non-null  int64  
 3   ANIO          249801 non-null  int64  
 4   HORA          249801 non-null  float64
 5   IMP_BOL       249801 non-null  float64
 6   DESCUENTO     249801 non-null  str    
 7   PUNTOS        249801 non-null  int64  
 8   DENUNCIANTE   249801 non-null  str    
 9   HECHO-BOL     249801 non-null  str    
 10  VEL_LIMITE    22182 non-null   Int64  
 11  VEL_CIRCULA   22182 non-null   Int64  
 12  COORDENADA-X  123882 non-null  float64
 13  COORDENADA-Y  123882 non-null  float64
dtypes: Int64(2), float64(4), int64(3), str(5)
memory usage: 27.2 MB


### Creación de columna nueva

__a)__ A partir del dataframe `multas` que contiene las columnas `MES`, `ANIO` y `HORA`, crear una nueva columna llamada `fecha` de tipo datetime que combine dicha información en un único objeto de fecha y hora. Usa como día el día 1.

In [26]:
# 1. Extraer las horas (la parte entera del número)
horas = multas['HORA'].astype(int)

# 2. Extraer los minutos (la parte decimal multiplicada por 100)
# Restar la parte entera y multiplicamos por 100. Usar round() para evitar
# problemas de precisión de los float (ej: 0.30000004 -> 30)
minutos = ((multas['HORA'] - horas) * 100).round().astype(int)

# 3. Construir la fecha usando un diccionario
multas['fecha'] = pd.to_datetime({
    'year': multas['ANIO'],
    'month': multas['MES'],
    'day': 1,      # El día 1 como pide el ejercicio
    'hour': horas,
    'minute': minutos
})

In [27]:
# Listar las columnas para comprobar que la hora encaja
multas[['ANIO', 'MES', 'HORA', 'fecha']].head()

,ANIO,MES,HORA,fecha
0,2024,12,20.23,2024-12-01 20:23:00
1,2024,12,20.27,2024-12-01 20:27:00
2,2024,12,20.45,2024-12-01 20:45:00
3,2024,12,16.30,2024-12-01 16:30:00
4,2024,12,16.50,2024-12-01 16:50:00


In [28]:
# Tipo de la nueva columna fecha
multas[['fecha']].dtypes

fecha    datetime64[us]
dtype: object

__b)__ Modifica el dataframe `multas` de forma que el índice sea el valor de la columna creada anteriormente.

In [29]:
# Reasignar variable
multas = multas.set_index('fecha')

In [30]:
multas.head()

,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
fecha,,,,,,,,,,,,,,
2024-12-01 20:23:00,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.,<NA>,<NA>,NaN,NaN
2024-12-01 20:27:00,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",<NA>,<NA>,NaN,NaN
2024-12-01 20:45:00,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACIÓN DE UN...,<NA>,<NA>,NaN,NaN
2024-12-01 16:30:00,LEVE,CL BRAVO MURILLO 24,12,2024,16.30,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.,<NA>,<NA>,NaN,NaN
2024-12-01 16:50:00,LEVE,CL BRAVO MURILLO 16,12,2024,16.50,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",<NA>,<NA>,NaN,NaN


In [31]:
multas.index

DatetimeIndex(['2024-12-01 20:23:00', '2024-12-01 20:27:00',
               '2024-12-01 20:45:00', '2024-12-01 16:30:00',
               '2024-12-01 16:50:00', '2024-12-01 17:06:00',
               '2024-12-01 17:12:00', '2024-12-01 18:01:00',
               '2024-12-01 18:44:00', '2024-12-01 20:21:00',
               ...
               '2024-12-01 18:21:00', '2024-12-01 19:07:00',
               '2024-12-01 20:18:00', '2024-12-01 21:21:00',
               '2024-12-01 22:17:00', '2024-12-01 10:40:00',
               '2024-12-01 08:13:00', '2024-12-01 08:20:00',
               '2024-12-01 12:13:00', '2024-12-01 12:51:00'],
              dtype='datetime64[us]', name='fecha', length=249801, freq=None)

__c)__ Elimina del dataframe la columna `fecha`.

`fecha` es el índice actual personalizado. Si se destruye, vuelve a ser una columna. Mediante `drop=True` se borran completamente los datos de `fecha`.

In [32]:
multas = multas.reset_index(drop=True)

In [33]:
multas.head()

,CALIFICACION,LUGAR,MES,ANIO,HORA,IMP_BOL,DESCUENTO,PUNTOS,DENUNCIANTE,HECHO-BOL,VEL_LIMITE,VEL_CIRCULA,COORDENADA-X,COORDENADA-Y
0,LEVE,CL CLARA DEL REY 36,12,2024,20.23,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.,<NA>,<NA>,NaN,NaN
1,LEVE,CL CLARA DEL REY 28,12,2024,20.27,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",<NA>,<NA>,NaN,NaN
2,GRAVE,CL CANILLAS 63,12,2024,20.45,200.0,SI,0,SER,ESTACIONAR OBSTACULIZANDO LA UTILIZACIÓN DE UN...,<NA>,<NA>,NaN,NaN
3,LEVE,CL BRAVO MURILLO 24,12,2024,16.30,60.0,SI,0,SER,ESTACIONAR CON AUTORIZACIÓN NO VÁLIDA.,<NA>,<NA>,NaN,NaN
4,LEVE,CL BRAVO MURILLO 16,12,2024,16.50,90.0,SI,0,SER,"ESTACIONAR, SIN LA CORRESPONDIENTE AUTORIZACIÓ...",<NA>,<NA>,NaN,NaN
